In [ ]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI

/Users/armaanjagirdar/Projects/ML_Project/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/armaanjagirdar/Projects/ML_Project/env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
load_dotenv()

True

In [5]:
graph = Neo4jGraph(url="neo4j://127.0.0.1:7687", username=os.getenv("NEO4J_USER"), password=os.getenv("NEO4J_PASSWORD"), enhanced_schema=True)

In [13]:
graph.refresh_schema()
print(graph.schema)

Node properties:
- **Entity**
  - `name`: STRING Example: "Alternaria Blotch"
Relationship properties:

The relationships:
(:Entity)-[:TREATED_BY]->(:Entity)
(:Entity)-[:AFFECTS]->(:Entity)
(:Entity)-[:PRESENTS]->(:Entity)
(:Entity)-[:HOST]->(:Entity)
(:Entity)-[:IS_HEALTHY]->(:Entity)
(:Entity)-[:CAUSED_BY]->(:Entity)
(:Entity)-[:AFFECTED_BY]->(:Entity)
(:Entity)-[:IS_A]->(:Entity)
(:Entity)-[:OCCURS_IN]->(:Entity)
(:Entity)-[:IS_PATHOGEN_OF]->(:Entity)


In [14]:
from langchain_neo4j.chains.graph_qa.cypher import GraphCypherQAChain

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    verbose=True,
    allow_dangerous_requests=True
)

In [17]:
question = "What all dieseases affect apples?"
result = cypher_chain.invoke({"query": question})
print("Answer:", result['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (disease:Entity)-[:AFFECTS]->(apple:Entity)
WHERE apple.name = "Apple"
RETURN disease.name

Full Context:
[{'disease.name': 'Alternaria Blotch'}, {'disease.name': 'Apple Alternaria Blotch'}, {'disease.name': 'Apple Scab'}, {'disease.name': 'Black Rot'}, {'disease.name': 'Brown Spot'}, {'disease.name': 'Cedar Apple Rust'}, {'disease.name': 'Frog Eye Leaf Spot'}, {'disease.name': 'Grey Spot'}, {'disease.name': 'Leaf Rust'}, {'disease.name': 'Mosaic Virus'}]

> Finished chain.
Answer: Alternaria Blotch, Apple Alternaria Blotch, Apple Scab, Black Rot, Brown Spot, Cedar Apple Rust, Frog Eye Leaf Spot, Grey Spot, Leaf Rust, and Mosaic Virus affect apples.
